In [3]:
import pandas as pd
from lxml import etree

In [ ]:
tree = etree.parse('datasets/eu-multibooked/4evora-hostel_1.kaf')
root = tree.getroot()

# Show all direct children of <NAF>
for child in root:
    print(child.tag, child.attrib, len(child))

# Peek at the <text> layer
text_layer = root.find('.//text')
print(etree.tostring(text_layer, pretty_print=True, encoding='unicode')[:1500])

nafHeader {} 4
text {} 21
terms {} 42
chunks {} 24
opinions {} 7
<text>
    <wf id="w1" offset="0" length="2" sent="1" para="1">Ez</wf>
    <wf id="w2" offset="3" length="4" sent="1" para="1">dago</wf>
    <wf id="w3" offset="8" length="10" sent="1" para="1">seinalerik</wf>
    <wf id="w4" offset="19" length="7" sent="1" para="1">bertara</wf>
    <wf id="w5" offset="27" length="8" sent="1" para="1">iristeko</wf>
    <wf id="w6" offset="36" length="4" sent="1" para="1">Leku</wf>
    <wf id="w7" offset="41" length="8" sent="1" para="1">modernoa</wf>
    <wf id="w8" offset="49" length="1" sent="1" para="1">,</wf>
    <wf id="w9" offset="51" length="6" sent="1" para="1">garbia</wf>
    <wf id="w10" offset="58" length="3" sent="1" para="1">eta</wf>
    <wf id="w11" offset="62" length="7" sent="1" para="1">txukuna</wf>
    <wf id="w12" offset="69" length="1" sent="1" para="1">.</wf>
    <wf id="w13" offset="71" length="9" sent="2" para="1">Langileak</wf>
    <wf id="w14" offset="81" length="

In [242]:
from collections import Counter
import glob, os

#used AI to correctly extract .kaf files (never used before)
def extract_with_label(path):
    root = etree.parse(path).getroot()

    # text (same as before)
    wfs = root.xpath('.//text/wf') or root.xpath('.//*[local-name()="wf"]')
    text = ' '.join((w.text or '').strip() for w in wfs)

    # polarities from all opinion_expression elements
    ops = root.xpath('.//*[local-name()="opinion_expression"]')
    polarities = [op.get('polarity') for op in ops if op.get('polarity')]

    if not polarities:
        label, majority, n_ops = None, None, 0
    else:
        counts = Counter(polarities)
        majority, _ = counts.most_common(1)[0]
        label, n_ops = majority, len(polarities)

    return {
        'file': os.path.basename(path),
        'review': text,
        'label': label,                    # majority polarity
        'polarities': polarities,          # raw list, for inspection
        'n_opinions': n_ops,
    }

df = pd.DataFrame([extract_with_label(f) for f in glob.glob('datasets/eu-multibooked/*.kaf')])
df.to_csv('reviews.csv', index=False, encoding='utf-8')
print(df.shape)

(343, 5)


## Basque csv

In [255]:
df_eu = pd.read_csv('reviews.csv')

In [256]:
df_eu.loc[df_eu['label']=='StrongPositive','label'] = 'Positive'
df_eu.loc[df_eu['label']=='StrongNegative','label'] = 'Negative'

In [257]:
df_eu['label'].value_counts()

label
Positive    294
Negative     48
Name: count, dtype: int64

In [258]:
import random
from deep_translator import GoogleTranslator
from deep_translator.exceptions import TooManyRequests
import time

translator = GoogleTranslator(source='eu', target='en')

def translate_with_retry(translator, text, max_retries=3): #got AI help here due to constant request error
    for attempt in range(max_retries):
        try:
            result = translator.translate(text)
            time.sleep(5 + random.uniform(0, 2))
            return result
        except TooManyRequests:
            wait = 30 * (attempt + 1)
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)
    return None

## can someone please try to run this my IP was blocked by google :'(

In [265]:
translator.translate(df_eu.iloc[0]['review']) #i need someone else to test this with different IP

TooManyRequests: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function

In [264]:
df_eu['review_en'] = df_eu['review'].apply(lambda t: translate_with_retry(translator, t)) #if previous work, try this

Rate limited, waiting 30s...


KeyboardInterrupt: 

## Spanish csv

In [205]:
df_es = pd.read_csv('datasets/IMDB Dataset SPANISH.csv')

In [206]:
df_es = df_es.drop(columns=['sentimiento','Unnamed: 0'])

In [207]:
df_es.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review_en  50000 non-null  object
 1   review_es  50000 non-null  object
 2   sentiment  50000 non-null  object
dtypes: object(3)
memory usage: 1.1+ MB


In [208]:
df_es['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [209]:
df_es = df_es.rename(columns={'review_es':'review', 'review_en':'translation_en'})

In [210]:
df_es.head()

,translation_en,review,sentiment
0,One of the other reviewers has mentioned that ...,Uno de los otros críticos ha mencionado que de...,positive
1,A wonderful little production. The filming tec...,Una pequeña pequeña producción.La técnica de f...,positive
2,I thought this was a wonderful way to spend ti...,Pensé que esta era una manera maravillosa de p...,positive
3,Basically there's a family where a little boy ...,"Básicamente, hay una familia donde un niño peq...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","El ""amor en el tiempo"" de Petter Mattei es una...",positive


---------------------

In [5]:
df_pt = pd.read_csv('datasets/PT-EN Zomato Dataset.csv')

In [212]:
df_pt.columns #too many with possible personal data and non relevant for case

Index(['review_id', 'user_pt_id', 'user_pt_name', 'user_in_id', 'user_in_name',
       'text', 'rating', 'created_at', 'res_id', 'res_name', 'res_location',
       'res_rating', 'res_price', 'res_zomato_id', 'res_lat', 'res_lng',
       'res_address_full', 'res_address_city', 'res_about', 'res_group_name',
       'res_group_type', 'res_priority', 'res_chain_id', 'res_menu_id',
       'res_et', 'res_time_slot', 'res_company_name', 'res_seats_exterior',
       'res_seats_interior', 'res_kitchen_Types', 'res_infos',
       'res_delivery_Methods', 'text_language', 'text_translation_pt',
       'text_translation_en'],
      dtype='object')

In [299]:
df_pt = df_pt[['review_id','text_translation_pt','text_translation_en','rating']].copy()
df_pt = df_pt.dropna()
df_pt = df_pt.sample(n=50000,random_state=42) #so is similar to spanish dataset
df_pt.to_csv('PT-EN Zomato cleaned.csv')

## Portuguese csv

In [6]:
df_pt = pd.read_csv('PT-EN Zomato cleaned.csv')
df_pt = df_pt.rename(columns={'text_translation_pt':'review','text_translation_en':'translation_en'})

In [7]:
def sentiment_labeling(stars):
    """
    Based on review stars (0-5) 
    """
    if stars <= 2.5:
        return 'negative'
    elif stars > 3.5:
        return 'positive'
    else:
        return 'neutral'

In [8]:
df_pt['sentiment'] = df_pt['rating'].apply(lambda s: sentiment_labeling(s))

In [9]:
df_pt['sentiment'].value_counts()

sentiment
positive    34309
neutral     10423
negative     5268
Name: count, dtype: int64

In [10]:
df_pt

,Unnamed: 0,review_id,review,translation_en,rating,sentiment
0,430273,5108432,Á procura de uma opção diferente no centro com...,Looking for a different option in the shopping...,3.5,neutral
1,82955,4782852,Brasserie que curioso que eu estava e que expe...,"Brasserie, how curious I was and what expectat...",5.0,positive
2,226016,4852265,Um bom sitio para petiscar e com uns dos melho...,A good place to have a bite to eat and with so...,4.5,positive
3,532418,4613665,Visitei o espaço na companhia da minha família...,I visited the space with my family to celebrat...,4.0,positive
4,323847,4857556,"Frequento este restaurante há anos. E, como ta...",I've been frequenting this restaurant for year...,4.5,positive
...,...,...,...,...,...,...
49995,588860,3693110,Fomos lá possivelmente no pior dia do ano para...,We went there on possibly the worst day of the...,2.5,negative
49996,1146998,4128302,Espaço super bonito e elegante. A comida muito...,Super cute and elegant space. The food was ver...,4.0,positive
49997,960677,3604263,Adoramos! Espaço bastante acolhedor com excele...,We love it! Very cozy space with excellent sna...,5.0,positive
49998,93910,4243202,Fiquei fã. Jantar perfeito. Optamos pelo menu ...,I became a fan. Perfect dinner. We opted for t...,4.5,positive


---------------------

In [ ]:
bad_char = '\ufffd' #spotted some [?] in text before

In [305]:
for d in [df_pt,df_es,df_eu]:
    print("review column has bad chars:", d['review'].str.contains(bad_char).sum())

review column has bad chars: 3
review column has bad chars: 0
review column has bad chars: 0


In [ ]:
df_pt['review'] = df_pt['review'].str.replace('\ufffd', '', regex=False)

In [224]:
print("PT column has bad chars:", df_pt['review'].str.contains(bad_char).sum()) #showed 3 errors in text

PT column has bad chars: 0


## dividing with and without stopwords for testing

In [231]:
import nltk
from nltk.corpus import stopwords

In [ ]:
#all languages have different stopwords so consider separatelly and then use single vectorizer
sw_pt = set(stopwords.words('portuguese')) - set(['não','nem','sem','muito'])
sw_es = set(stopwords.words('spanish')) - set(['no','muy','poco','sin'])
sw_eu = set(stopwords.words('basque')) - set(['oso','biziki','ez'])
sw_em = set(stopwords.words('english')) - set(['not','very','so',"don't",'no'])

In [ ]:
df_pt_sw = df_pt.copy()
df_pt_sw['review'] = df_pt['review'].apply(lambda x: " ".join(x for x in x.split() if x not in sw_pt))

In [287]:
df_es_sw = df_es.copy()
df_es_sw['review'] = df_es['review'].apply(lambda x: " ".join(x for x in x.split() if x not in sw_es))

In [288]:
df_eu_sw = df_eu.copy()
df_eu_sw['review'] = df_eu['review'].apply(lambda x: " ".join(x for x in x.split() if x not in sw_eu))

## Vectoriser for pipelines

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#create vectoriser for models pipeline
vectorizer = TfidfVectorizer(lowercase=True, #skip preprocessing it in previos steps and include here
                             strip_accents='unicode', #unicode has more characters that can be found in protuguese and basque
                             analyzer='words') #we can test the others too

## Next steps:
+ fix basque/euskera translation and add labels
+ Have a baseline model (linear for classificatio)
+ have more advanced models to compare (like svm and knn)
+ train english model with best found model (we should add another dataset for this?)
+ test the performance on translated_en column agains original review column
+ important: it is expected that second phase of project has better results with added context specially for basque